# 05 — Source Independence & RedundancyAre any sources measuring the same thing? Correlated sources double-countin the CTI. We find redundancy groups and compute effective weights.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.decomposition import PCAfrom sklearn.feature_selection import mutual_info_classifdf = pd.read_parquet("daily_matrix.parquet")sources = [c for c in df.columns if c not in ("n_yellow", "is_elevated", "total")           and not c.startswith("cat_")]# Drop sources with zero varianceactive = [s for s in sources if df[s].std() > 0]X = df[active]y = df["is_elevated"].valuesprint(f"{len(active)} active sources, {len(df)} days")

## Correlation matrix

In [ ]:
corr = X.corr()fig, ax = plt.subplots(figsize=(12, 10))mask = np.triu(np.ones_like(corr, dtype=bool))sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",            center=0, vmin=-1, vmax=1, ax=ax, square=True)ax.set_title("Source Correlation Matrix")plt.tight_layout()plt.savefig("correlation_matrix.png", dpi=150)plt.show()# High correlationsprint("Correlated pairs (|r| > 0.3):\n")for i, sa in enumerate(active):    for sb in active[i+1:]:        r = corr.loc[sa, sb]        if abs(r) > 0.3:            rel = "redundant" if abs(r) > 0.7 else "correlated"            print(f"  {sa:20s} ↔ {sb:20s}  r={r:+.3f}  ({rel})")

## Mutual information with YELLOW statusHow much does each source tell us about whether a day is elevated?

In [ ]:
mi = mutual_info_classif(X.fillna(0), y, random_state=42)mi_series = pd.Series(mi, index=active).sort_values(ascending=False)fig, ax = plt.subplots(figsize=(10, 8))mi_series.plot.barh(ax=ax, color="steelblue")ax.set_title("Mutual Information: Source → YELLOW Status")ax.set_xlabel("MI (bits)")plt.tight_layout()plt.savefig("mutual_information.png", dpi=150)plt.show()print("Mutual information ranking:")print(mi_series.round(4).to_string())

## PCA — how many independent signals do we actually have?

In [ ]:
from sklearn.preprocessing import StandardScalerX_scaled = StandardScaler().fit_transform(X.fillna(0))pca = PCA()pca.fit(X_scaled)explained = pca.explained_variance_ratio_cumulative = np.cumsum(explained)fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))ax1.bar(range(1, len(explained)+1), explained, color="steelblue")ax1.set_xlabel("Component")ax1.set_ylabel("Variance Explained")ax1.set_title("PCA Explained Variance")ax2.plot(range(1, len(cumulative)+1), cumulative, "bo-")ax2.axhline(0.90, color="red", linestyle="--", label="90%")ax2.axhline(0.95, color="orange", linestyle="--", label="95%")ax2.set_xlabel("Components")ax2.set_ylabel("Cumulative Variance")ax2.set_title("Cumulative Explained Variance")ax2.legend()plt.tight_layout()plt.savefig("pca_variance.png", dpi=150)plt.show()n_90 = np.argmax(cumulative >= 0.90) + 1n_95 = np.argmax(cumulative >= 0.95) + 1print(f"Components for 90% variance: {n_90} (out of {len(active)} sources)")print(f"Components for 95% variance: {n_95}")print(f"\n→ {len(active)} sources contain only ~{n_90} independent signals")# Top loadings for PC1 and PC2loadings = pd.DataFrame(pca.components_[:3].T, index=active, columns=["PC1", "PC2", "PC3"])print(f"\nPCA loadings (top 3 components):")print(loadings.round(3).to_string())

## Effective weights (MI-weighted, independence-adjusted)

In [ ]:
CURRENT = {"gpsjam": 20, "adsb": 15, "acled": 15, "firms": 15,           "ais": 10, "telegram": 10, "rss": 5, "gdelt": 5, "ioda": 5}# For each source, discount by max correlation with a higher-MI sourceeffective = {}mi_sorted = mi_series.index.tolist()for src in active:    base_mi = mi_series.get(src, 0)    max_corr = 0    for other in mi_sorted:        if other == src: continue        if mi_series.get(other, 0) <= base_mi: continue        r = abs(corr.loc[src, other]) if src in corr.index and other in corr.columns else 0        max_corr = max(max_corr, r)    independence = max(1.0 - max_corr, 0.2)  # floor at 20%    effective[src] = base_mi * independenceeff_series = pd.Series(effective).sort_values(ascending=False)eff_norm = (eff_series / eff_series.sum() * 100).round(1)comparison = pd.DataFrame({    "MI_weight": (mi_series / mi_series.sum() * 100).round(1),    "effective_weight": eff_norm,    "current_weight": pd.Series(CURRENT),}).fillna(0).sort_values("effective_weight", ascending=False)print("\n══════════════════════════════════════")print("  INDEPENDENCE-ADJUSTED WEIGHTS")print("══════════════════════════════════════\n")print(comparison.to_string())fig, ax = plt.subplots(figsize=(10, 6))comparison.head(12).plot.bar(ax=ax)ax.set_title("Source Weights: MI vs Effective vs Current")ax.set_ylabel("Weight (%)")plt.tight_layout()plt.savefig("effective_weights.png", dpi=150)plt.show()